In [30]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

Define Paths

In [31]:
RAW_DATA_PATH = r"C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\raw"
PROCESSED_DATA_PATH = r"C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\cleaned"

print("Raw Data Path :", RAW_DATA_PATH)
print("Processed Path :", PROCESSED_DATA_PATH)

Raw Data Path : C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\raw
Processed Path : C:\Users\mayank\OneDrive\Desktop\Project-FORESIGHT\Data\cleaned


Load Data

In [32]:
sales = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_sales.csv")

inventory = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_inventory.csv")

sku = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_skus.csv")

stores = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_stores.csv")

customers = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_customers.csv")

promotions = pd.read_csv(RAW_DATA_PATH + "\\" + "bm_promotions.csv")

Data Shape

In [33]:
datasets = {
    "Sales": sales,
    "Inventory": inventory,
    "SKU": sku,
    "Stores": stores,
    "Customers": customers,
    "Promotions": promotions
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Sales: (641843, 9)
Inventory: (8735, 7)
SKU: (200, 7)
Stores: (50, 5)
Customers: (5000, 7)
Promotions: (33, 6)


Rename Customer key

In [34]:
if "cust_id" in customers.columns:
    customers.rename(columns={"cust_id": "customer_id"}, inplace=True)

print(customers.columns)

Index(['customer_id', 'age', 'gender', 'city', 'loyalty_segment',
       'preferred_channel', 'registration_date'],
      dtype='object')


Remove Duplicate

In [35]:
for name, df in datasets.items():

    before = df.shape[0]

    df.drop_duplicates(inplace=True)

    after = df.shape[0]

    print(f"{name}")

    print("Removed:", before-after)

Sales
Removed: 45
Inventory
Removed: 0
SKU
Removed: 0
Stores
Removed: 0
Customers
Removed: 0
Promotions
Removed: 0


Missing Value Report

In [36]:
for name, df in datasets.items():

    print("="*50)

    print(name)

    print(df.isnull().sum())

Sales
date                 0
store_id             0
sku_id               0
customer_id     159777
quantity             0
unit_price           0
total_value          0
channel              0
discount_pct         0
dtype: int64
Inventory
store_id             0
sku_id               0
stock_on_hand        0
reorder_point        0
safety_stock         0
last_restock_date    0
snapshot_date        0
dtype: int64
SKU
sku_id         0
sku_name       0
category       0
subcategory    0
unit_price     0
cost_price     0
brand          0
dtype: int64
Stores
store_id        0
store_name      0
city            0
store_type      0
opening_date    0
dtype: int64
Customers
customer_id          0
age                  0
gender               0
city                 0
loyalty_segment      0
preferred_channel    0
registration_date    0
dtype: int64
Promotions
promo_name      0
start_date      0
end_date        0
discount_pct    0
promo_type      0
promo_id        0
dtype: int64


Fill Numeric Value

In [37]:
for name, df in datasets.items():

    numeric_columns = df.select_dtypes(include=np.number).columns

    for col in numeric_columns:

        df[col].fillna(df[col].median(), inplace=True)

C:\Users\mayank\AppData\Local\Temp\ipykernel_20488\3740879478.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


Fill Categorical Value

In [38]:
for name, df in datasets.items():

    categorical_columns = df.select_dtypes(include="object").columns

    for col in categorical_columns:

        df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\mayank\AppData\Local\Temp\ipykernel_20488\352862384.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)
C:\Users\mayank\AppData\Local\Temp\ipykernel_20488\352862384.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exam

Convert Date column

In [39]:
for name, df in datasets.items():

    for col in df.columns:

        if "date" in col.lower():

            df[col] = pd.to_datetime(df[col], errors="coerce")

Standardize Text Columns

In [40]:
for name, df in datasets.items():

    text_columns = df.select_dtypes(include="object").columns

    for col in text_columns:

        df[col] = df[col].str.strip()

        df[col] = df[col].str.title()

Remove Negative Quantity

In [41]:
if "quantity" in sales.columns:

    sales = sales[sales["quantity"] >= 0]

Remove Negative Sales Amount

In [42]:
if "sales_amount" in sales.columns:

    sales = sales[sales["sales_amount"] >= 0]

Remove Negative Inventory

In [43]:
if "stock_quantity" in inventory.columns:

    inventory = inventory[inventory["stock_quantity"] >= 0]

Outlier Detection (IQR)

In [44]:
def remove_outliers(df):

    numeric = df.select_dtypes(include=np.number).columns

    for col in numeric:

        Q1 = df[col].quantile(0.25)

        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR

        upper = Q3 + 1.5 * IQR

        df = df[(df[col] >= lower) & (df[col] <= upper)]

    return df

Apply Outlier Removal

In [45]:
sales = remove_outliers(sales)

inventory = remove_outliers(inventory)

Merge SKU Information

In [46]:
if "sku_id" in sales.columns and "sku_id" in sku.columns:

    sales = sales.merge(

        sku,

        on="sku_id",

        how="left"
    )

Merge Store Information

In [47]:
if "store_id" in sales.columns and "store_id" in stores.columns:

    sales = sales.merge(

        stores,

        on="store_id",

        how="left"
    )

Merge Customer Information

In [48]:
if "customer_id" in sales.columns and "customer_id" in customers.columns:

    sales = sales.merge(

        customers,

        on="customer_id",

        how="left"
    )

Merge Promotions

In [49]:
if "promotion_id" in sales.columns and "promotion_id" in promotions.columns:

    sales = sales.merge(

        promotions,

        on="promotion_id",

        how="left"
    )

Merge Inventory

In [50]:
# Check duplicate SKUs
print("Rows:", len(inventory))
print("Unique SKU:", inventory["sku_id"].nunique())

# Keep only one row per SKU
inventory_unique = inventory.drop_duplicates(subset="sku_id", keep="first")

print("Rows after:", len(inventory_unique))

sales = sales.merge(
    inventory_unique,
    on="sku_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_inventory")
)

Rows: 8474
Unique SKU: 200
Rows after: 200


Final Dataset Information

In [51]:
print(sales.shape)

sales.head()

(515212, 31)


,date,store_id,sku_id,customer_id,quantity,unit_price_x,total_value,channel,discount_pct,sku_name,category,subcategory,unit_price_y,cost_price,brand,store_name,city_x,store_type,opening_date,age,gender,city_y,loyalty_segment,preferred_channel,registration_date,store_id_inventory,stock_on_hand,reorder_point,safety_stock,last_restock_date,snapshot_date
0,2021-01-01,26,1124,2961.0,1,20.08,20.08,Store,15.0,Dairy_Cheese_1124,Dairy,Cheese,23.62,13.62,Premium,Bluemart Store 26,Abu Dhabi,High Street,2017-06-01 08:11:42.439024392,50,Female,Dubai,Silver,Website,2024-12-19,10,185,70,35,2025-09-27,2025-10-31
1,2021-01-01,38,1088,2507.0,1,17.99,17.99,Website,15.0,Household_Cleaning Supplies_1088,Household,Cleaning Supplies,21.16,15.32,Bluemart,Bluemart Store 38,Dubai,High Street,2017-09-16 04:05:51.219512196,60,Male,Dubai,Gold,Mobileapp,2023-12-17,9,241,96,48,2025-09-21,2025-10-31
2,2021-01-01,2,1093,1252.0,1,7.98,7.98,Store,0.0,Snacks_Chips_1093,Snacks,Chips,7.98,5.29,Bluemart,Bluemart Store 02,Sharjah,High Street,2017-07-01 00:00:00.000000000,40,Female,Dubai,Gold,Mobileapp,2024-11-20,9,289,125,62,2025-09-17,2025-10-31
3,2021-01-01,25,1067,1286.0,1,42.13,42.13,Mobileapp,15.0,Grocery_Cereals_1067,Grocery,Cereals,49.57,32.38,Premium,Bluemart Store 25,Dubai,High Street,2017-05-23 10:32:11.707317074,25,Male,Sharjah,Silver,Store,2024-04-02,11,303,116,58,2025-09-27,2025-10-31
4,2021-01-01,2,1043,2507.0,3,19.93,59.79,Mobileapp,15.0,Electronics_Batteries_1043,Electronics,Batteries,23.45,15.63,Budget,Bluemart Store 02,Sharjah,High Street,2017-07-01 00:00:00.000000000,60,Male,Dubai,Gold,Mobileapp,2023-12-17,9,137,47,23,2025-08-14,2025-10-31


Remaining Missing Values

In [52]:
sales.isnull().sum().sort_values(ascending=False)

date                  0
city_x                0
last_restock_date     0
safety_stock          0
reorder_point         0
stock_on_hand         0
store_id_inventory    0
registration_date     0
preferred_channel     0
loyalty_segment       0
city_y                0
gender                0
age                   0
opening_date          0
store_type            0
store_name            0
store_id              0
brand                 0
cost_price            0
unit_price_y          0
subcategory           0
category              0
sku_name              0
discount_pct          0
channel               0
total_value           0
unit_price_x          0
quantity              0
customer_id           0
sku_id                0
snapshot_date         0
dtype: int64

Fill Remaining Missing Values

In [53]:
sales.fillna(method="ffill", inplace=True)

sales.fillna(method="bfill", inplace=True)

C:\Users\mayank\AppData\Local\Temp\ipykernel_20488\708428985.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sales.fillna(method="ffill", inplace=True)
C:\Users\mayank\AppData\Local\Temp\ipykernel_20488\708428985.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sales.fillna(method="bfill", inplace=True)


Final Duplicate Check

In [54]:
print("Duplicates:", sales.duplicated().sum())

Duplicates: 0


Save Cleaned Sales

In [55]:
sales.to_csv(

    PROCESSED_DATA_PATH + "cleaned_sales.csv",

    index=False
)

print("cleaned_sales.csv Saved")

cleaned_sales.csv Saved


Save Cleaned Inventory

In [56]:
inventory.to_csv(

    PROCESSED_DATA_PATH + "\\" + "cleaned_inventory.csv",

    index=False
)

print("cleaned_inventory.csv Saved")

cleaned_inventory.csv Saved


Save Final Dataset

In [57]:
sales.to_csv(

    PROCESSED_DATA_PATH + "final_dataset.csv",

    index=False
)

print("final_dataset.csv Saved")

final_dataset.csv Saved


Cleaning Summary

In [58]:
summary = pd.DataFrame({

    "Dataset": [
        "Sales",
        "Inventory",
        "Final Dataset"
    ],

    "Rows": [
        sales.shape[0],
        inventory.shape[0],
        sales.shape[0]
    ],

    "Columns": [
        sales.shape[1],
        inventory.shape[1],
        sales.shape[1]
    ]
})

summary

,Dataset,Rows,Columns
0,Sales,515212,31
1,Inventory,8474,7
2,Final Dataset,515212,31
